# Health-Personalized Food Recommender System
**Dataset:** foodRecSys-V1 (Allrecipes.com)  
**Source:** https://www.kaggle.com/datasets/elisaxxygao/foodrecsysv1  

> **Files needed** — place in a `data/` folder next to this notebook:
> - `core-data_recipe.json` — recipe details (name, ingredients, nutrition, categories)
> - `core-data_train.csv` — user interactions (user_id, recipe_id, rating, date)
> - `core-data_test.csv` — held-out test interactions

**Why this dataset is better than Food.com for a recommender:**
- Interactions are binary implicit feedback (any interaction = strong interest)
- No positivity bias — 70% five-star problem does not exist
- Widely used in published RecSys research — results are comparable to literature
- 52,821 recipes | 68,768 users | 1,093,845 interactions

| Section | Work Package |
|---------|-------------|
| 1. Setup & Load | — |
| 2. Data Scraping — USDA API | WP: Data Scraping |
| 3. Data Cleaning | WP: Data Quality |
| 4. EDA | — |
| 5. Annotation | WP: Data Annotation |
| 6. Embeddings | WP: Vector Embeddings |
| 7. Recommender System | WP: Recommender System |
| 8. Evaluation | WP: Performance Evaluation |
| 9. Hyperparameter Tuning | WP: Hyperparameter Tuning |
| 10. Experiment Logging | WP: Experiments Logging |
| 11. Perturbation Analysis | WP: Perturbation Analysis |
| 12. Frontend | WP: Frontend Application |


---
## 1. Setup & Load


In [ ]:
import sys
!{sys.executable} -m pip install -q pandas numpy scikit-learn matplotlib seaborn \
    scikit-surprise optuna wandb requests anthropic
print('Done')


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, json, time, random, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
import requests

plt.rcParams.update({
    'figure.facecolor':'white','axes.facecolor':'#F8F7F4',
    'axes.grid':True,'grid.color':'#E0DED8',
    'axes.spines.top':False,'axes.spines.right':False,
})
C_BEFORE='#A32D2D'; C_AFTER='#0F6E56'; C_FLAG='#BA7517'; C_PURPLE='#534AB7'

os.makedirs('data',   exist_ok=True)
os.makedirs('plots',  exist_ok=True)
os.makedirs('models', exist_ok=True)
np.random.seed(42); random.seed(42)
print('Setup complete')


In [ ]:
# ── Load recipe JSON ──────────────────────────────────────────────────────────
# core-data_recipe.json contains recipe details
with open('data/core-data_recipe.json', 'r', encoding='utf-8') as f:
    recipes_raw = json.load(f)

print(f'Raw recipes loaded: {len(recipes_raw)}')
print('Sample keys:', list(recipes_raw[list(recipes_raw.keys())[0]].keys()) if isinstance(recipes_raw, dict) else list(recipes_raw[0].keys()))


In [ ]:
# ── Parse recipe JSON into DataFrame ─────────────────────────────────────────
# foodRecSys-V1 JSON structure: {recipe_id: {name, ingredients, nutritions, category, ...}}
rows = []
for rid, info in recipes_raw.items():
    row = {'recipe_id': str(rid), 'name': info.get('name','')}

    # Nutrition — stored as dict e.g. {'calories':'367 cal', 'fat':'22g', ...}
    nutr = info.get('nutritions', info.get('nutrition', {}))
    if isinstance(nutr, dict):
        def parse_num(s):
            if s is None: return np.nan
            import re
            m = re.search(r'[\d.]+', str(s))
            return float(m.group()) if m else np.nan

        row['calories']    = parse_num(nutr.get('calories',    nutr.get('Calories')))
        row['protein_g']   = parse_num(nutr.get('protein',     nutr.get('Protein')))
        row['carbs_g']     = parse_num(nutr.get('carbohydrates',nutr.get('carbs',nutr.get('Carbohydrates'))))
        row['fat_g']       = parse_num(nutr.get('fat',         nutr.get('Fat')))
        row['sodium_mg']   = parse_num(nutr.get('sodium',      nutr.get('Sodium')))
        row['fiber_g']     = parse_num(nutr.get('fiber',       nutr.get('Fiber')))
        row['sugar_g']     = parse_num(nutr.get('sugar',       nutr.get('Sugar')))
        row['sat_fat_g']   = parse_num(nutr.get('saturated fat',nutr.get('Saturated Fat')))

    # Category / cuisine
    cats = info.get('category', info.get('categories', []))
    if isinstance(cats, list): row['category'] = cats[0] if cats else 'unknown'
    else: row['category'] = str(cats)

    # Ingredients
    ingr = info.get('ingredients', [])
    row['ingredients'] = ingr if isinstance(ingr, list) else []
    row['n_ingredients'] = len(row['ingredients'])

    # Cook time
    row['minutes'] = parse_num(info.get('total_time', info.get('cook_time', info.get('cookTime'))))

    rows.append(row)

df_recipes = pd.DataFrame(rows)
print(f'Recipes parsed: {len(df_recipes)}')
print(f'Columns: {list(df_recipes.columns)}')
print(df_recipes[['name','calories','protein_g','carbs_g','fat_g']].head(5).to_string(index=False))


In [ ]:
# ── Load interaction data ─────────────────────────────────────────────────────
# The dataset provides pre-split train/test files
# Format: user_id, recipe_id, rating (or just interactions)

def load_interactions(path):
    df = pd.read_csv(path)
    print(f'  {path}: {len(df):,} rows  columns: {list(df.columns)}')
    return df

df_train_raw = load_interactions('data/core-data_train.csv')
df_test_raw  = load_interactions('data/core-data_test.csv')

print(f'\nTrain sample:')
print(df_train_raw.head(3).to_string(index=False))


In [ ]:
# ── Standardise column names ──────────────────────────────────────────────────
# Rename to standard names regardless of original column names
def standardise_interactions(df):
    df = df.copy()
    # Map common column name variants
    rename_map = {}
    for col in df.columns:
        cl = col.lower().strip()
        if cl in ['user_id','userid','user']:    rename_map[col] = 'user_id'
        elif cl in ['recipe_id','recipeid','item_id','itemid','recipe']: rename_map[col] = 'recipe_id'
        elif cl in ['rating','score','interaction']: rename_map[col] = 'rating'
        elif cl in ['date','timestamp','time']:   rename_map[col] = 'date'
    df = df.rename(columns=rename_map)
    df['user_id']   = df['user_id'].astype(str)
    df['recipe_id'] = df['recipe_id'].astype(str)
    # Binary implicit feedback — any interaction = 1
    if 'rating' not in df.columns:
        df['rating'] = 1
    else:
        df['rating'] = 1   # treat all interactions as positive
    return df

df_train_inter = standardise_interactions(df_train_raw)
df_test_inter  = standardise_interactions(df_test_raw)

# Keep only recipes that exist in recipe JSON
valid_ids = set(df_recipes['recipe_id'].astype(str))
df_train_inter = df_train_inter[df_train_inter['recipe_id'].isin(valid_ids)].reset_index(drop=True)
df_test_inter  = df_test_inter[df_test_inter['recipe_id'].isin(valid_ids)].reset_index(drop=True)

print(f'Train interactions: {len(df_train_inter):,}')
print(f'Test  interactions: {len(df_test_inter):,}')
print(f'Unique users (train): {df_train_inter["user_id"].nunique():,}')
print(f'Unique recipes (train): {df_train_inter["recipe_id"].nunique():,}')


---
## 2. Data Scraping — USDA API
**Work Package: Data Scraping**

foodRecSys-V1 already includes nutritional data from Allrecipes. We enrich
a sample of recipes with additional nutrients from the USDA FoodData Central API
(`fiber_g`, `potassium_mg`, `calcium_mg`, `iron_mg`) that are not in the original dataset.

**Get a free API key at:** https://fdc.nal.usda.gov/api-key-signup.html


In [ ]:
USDA_API_KEY = 'DEMO_KEY'   # Replace with your free key
USDA_BASE    = 'https://api.nal.usda.gov/fdc/v1'

def scrape_usda(food_name):
    try:
        r = requests.get(f'{USDA_BASE}/foods/search',
            params={'query':food_name,'api_key':USDA_API_KEY,
                    'pageSize':1,'dataType':'Foundation,SR Legacy'},timeout=8)
        r.raise_for_status()
        foods = r.json().get('foods',[])
        if not foods: return {}
        nm = {n['nutrientName']:n['value'] for n in foods[0].get('foodNutrients',[])}
        return {
            'fiber_g':      nm.get('Fiber, total dietary', np.nan),
            'potassium_mg': nm.get('Potassium, K',         np.nan),
            'calcium_mg':   nm.get('Calcium, Ca',          np.nan),
            'iron_mg':      nm.get('Iron, Fe',             np.nan),
        }
    except: return {}

SCRAPE_LIVE = False   # Set True only for first-time scraping

if SCRAPE_LIVE:
    print('Scraping USDA API for 500 recipes...')
    sample = df_recipes.sample(n=min(500, len(df_recipes)), random_state=42)[['recipe_id','name']]
    rows_usda = []
    for i, (_, row) in enumerate(sample.iterrows()):
        if i % 100 == 0: print(f'  {i}/{len(sample)}')
        rec = scrape_usda(row['name']); rec['recipe_id'] = row['recipe_id']
        rows_usda.append(rec); time.sleep(0.4)
    df_usda = pd.DataFrame(rows_usda)
    df_usda.to_csv('data/usda_enrichment.csv', index=False)
    print(f'Saved {len(df_usda)} records')
elif os.path.exists('data/usda_enrichment.csv'):
    df_usda = pd.read_csv('data/usda_enrichment.csv')
    df_usda['recipe_id'] = df_usda['recipe_id'].astype(str)
    print(f'Loaded existing USDA enrichment: {len(df_usda)} records')
else:
    print('No enrichment file. Set SCRAPE_LIVE=True to scrape.')
    df_usda = pd.DataFrame(columns=['recipe_id','fiber_g','potassium_mg','calcium_mg','iron_mg'])

# Merge — only fill missing fiber values (dataset may already have fiber_g)
usda_cols = ['fiber_g','potassium_mg','calcium_mg','iron_mg']
for col in usda_cols:
    if col not in df_recipes.columns:
        df_recipes[col] = np.nan

if len(df_usda) > 0:
    df_usda['recipe_id'] = df_usda['recipe_id'].astype(str)
    for col in usda_cols:
        if col in df_usda.columns:
            mapping = df_usda.set_index('recipe_id')[col]
            mask = df_recipes['recipe_id'].isin(mapping.index) & df_recipes[col].isna()
            df_recipes.loc[mask, col] = df_recipes.loc[mask, 'recipe_id'].map(mapping)

print(f'Fiber coverage: {df_recipes["fiber_g"].notna().sum()} / '
      f'{len(df_recipes)} ({df_recipes["fiber_g"].notna().mean():.1%})')


---
## 3. Data Cleaning
**Work Package: Data Quality**

**Cook time: 10–180 min** — restaurant-style meals only.
Excludes drinks/instant items (< 10 min) and home slow-cooks (> 3 hours).


In [ ]:
CORE_COLS = ['calories','protein_g','carbs_g','fat_g','sodium_mg','sugar_g','minutes']

# ── EDA BEFORE ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

miss = df_recipes[CORE_COLS].isna().mean().sort_values(ascending=False)
bars = axes[0].barh(miss.index, miss.values*100,
                     color=[C_BEFORE if v>0.05 else C_AFTER for v in miss],
                     edgecolor='white')
axes[0].axvline(5, color=C_FLAG, linestyle='--', linewidth=1, label='5% threshold')
axes[0].set_xlabel('Missing (%)'); axes[0].legend(fontsize=8)
axes[0].set_title('Missing Values — BEFORE', fontweight='bold', color=C_BEFORE)
axes[0].bar_label(bars, fmt='%.1f%%', padding=3, fontsize=8)

cal = df_recipes['calories'].dropna()
axes[1].hist(cal, bins=100, color=C_BEFORE, alpha=0.8, edgecolor='none')
axes[1].axvline(2000, color=C_FLAG, linestyle='--', linewidth=1.5, label='Cutoff:2000')
axes[1].set_xlabel('Calories'); axes[1].legend(fontsize=8)
axes[1].set_title('Calories — BEFORE', fontweight='bold', color=C_BEFORE)
axes[1].text(0.97, 0.88, f'Above 2000: {(cal>2000).sum():,}',
              transform=axes[1].transAxes, ha='right', fontsize=8,
              color=C_BEFORE, fontweight='bold')

mins = df_recipes['minutes'].dropna()
axes[2].hist(mins.clip(upper=400), bins=100, color=C_BEFORE, alpha=0.8, edgecolor='none')
axes[2].axvline(10,  color=C_AFTER, linestyle='--', linewidth=1.5, label='Min:10')
axes[2].axvline(180, color=C_FLAG,  linestyle='--', linewidth=1.5, label='Max:180')
axes[2].set_xlabel('Cook time (min)'); axes[2].legend(fontsize=8)
axes[2].set_title('Cook Time — BEFORE', fontweight='bold', color=C_BEFORE)
n_bad = ((mins<10)|(mins>180)).sum()
axes[2].text(0.97, 0.88, f'Outside [10,180]: {n_bad:,}',
              transform=axes[2].transAxes, ha='right', fontsize=8,
              color=C_BEFORE, fontweight='bold')

plt.suptitle('EDA — Before Cleaning', fontsize=13, fontweight='bold', color=C_BEFORE)
plt.tight_layout()
plt.savefig('plots/eda_before.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Apply cleaning rules ───────────────────────────────────────────────────────
VALID_RANGES = {
    'calories':  (50,  2000),
    'protein_g': (0,   150),
    'carbs_g':   (0,   300),
    'fat_g':     (0,   150),
    'sodium_mg': (0,  5000),
    'sugar_g':   (0,   200),
    'minutes':   (10,  180),   # restaurant-style only
}

df = df_recipes.copy()
n0 = len(df)

# 1. Range violations
flag = pd.Series(False, index=df.index)
for col, (lo, hi) in VALID_RANGES.items():
    if col in df.columns:
        flag |= df[col].notna() & ((df[col] < lo) | (df[col] > hi))
df = df[~flag].reset_index(drop=True)
print(f'Range violations removed:      {n0 - len(df):,}')

# 2. Duplicate names
n1 = len(df)
df = df.drop_duplicates(subset=['name'], keep='first').reset_index(drop=True)
print(f'Duplicate names removed:       {n1 - len(df):,}')

# 3. Mean impute missing nutritional values
print('Mean imputation:')
for col in list(VALID_RANGES.keys()) + ['sat_fat_g','fiber_g','potassium_mg','calcium_mg','iron_mg']:
    if col in df.columns and df[col].isna().sum() > 0:
        fill = df[col].mean()
        df[col] = df[col].fillna(fill)
        print(f'  {col:<18} → mean = {fill:.2f}')

# 4. Keep only clean recipes in interactions
clean_ids = set(df['recipe_id'].astype(str))
df_train  = df_train_inter[df_train_inter['recipe_id'].isin(clean_ids)].copy().reset_index(drop=True)
df_test   = df_test_inter[df_test_inter['recipe_id'].isin(clean_ids)].copy().reset_index(drop=True)

# Remove duplicate (user, recipe) pairs — keep first
df_train = df_train.drop_duplicates(subset=['user_id','recipe_id'], keep='first').reset_index(drop=True)
df_test  = df_test.drop_duplicates(subset=['user_id','recipe_id'], keep='first').reset_index(drop=True)

print(f'\nRecipes:          {n0:,} → {len(df):,}')
print(f'Train interactions: {len(df_train):,}')
print(f'Test  interactions: {len(df_test):,}')

df.to_csv('data/recipes_clean.csv', index=False)
df_train.to_csv('data/interactions_train.csv', index=False)
df_test.to_csv('data/interactions_test.csv', index=False)
print('Saved.')


---
## 4. EDA After Cleaning


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

miss_a = df[CORE_COLS].isna().mean().sort_values(ascending=False)
bars = axes[0].barh(miss_a.index, miss_a.values*100, color=C_AFTER, edgecolor='white')
axes[0].set_xlabel('Missing (%)'); axes[0].set_title('Missing — AFTER', fontweight='bold', color=C_AFTER)
axes[0].bar_label(bars, fmt='%.1f%%', padding=3, fontsize=8)

axes[1].hist(df_recipes['calories'].dropna().clip(upper=2500), bins=80,
              color=C_BEFORE, alpha=0.5, label='Before', edgecolor='none')
axes[1].hist(df['calories'].clip(upper=2500), bins=80,
              color=C_AFTER, alpha=0.7, label='After', edgecolor='none')
axes[1].set_xlabel('Calories'); axes[1].set_title('Calories Before vs After', fontweight='bold')
axes[1].legend(fontsize=8)

axes[2].hist(df_recipes['minutes'].dropna().clip(0, 250), bins=80,
              color=C_BEFORE, alpha=0.5, label='Before', edgecolor='none')
axes[2].hist(df['minutes'].clip(0, 250), bins=80,
              color=C_AFTER, alpha=0.7, label='After', edgecolor='none')
axes[2].axvline(10,  color='black', linestyle='--', linewidth=1, alpha=0.6)
axes[2].axvline(180, color='black', linestyle='--', linewidth=1, alpha=0.6)
axes[2].set_xlabel('Cook time (min)'); axes[2].set_title('Cook Time Before vs After', fontweight='bold')
axes[2].legend(fontsize=8)

plt.suptitle('EDA — After Cleaning', fontsize=13, fontweight='bold', color=C_AFTER)
plt.tight_layout()
plt.savefig('plots/eda_after.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Final: {len(df):,} recipes | {len(df_train):,} train | {len(df_test):,} test interactions')
print(df[CORE_COLS].describe().round(1).to_string())


In [ ]:
# ── Interaction sparsity ──────────────────────────────────────────────────────
n_users   = df_train['user_id'].nunique()
n_recipes = df_train['recipe_id'].nunique()
density   = len(df_train) / (n_users * n_recipes)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

rpu = df_train.groupby('user_id').size()
axes[0].hist(rpu.clip(upper=50), bins=50, color=C_PURPLE, alpha=0.8, edgecolor='none')
axes[0].set_xlabel('Interactions per user (clipped at 50)')
axes[0].set_title('User Activity', fontweight='bold')
axes[0].text(0.7, 0.85, f'Median: {rpu.median():.0f}', transform=axes[0].transAxes, fontsize=9)

rpr = df_train.groupby('recipe_id').size()
axes[1].hist(rpr.clip(upper=50), bins=50, color=C_AFTER, alpha=0.8, edgecolor='none')
axes[1].set_xlabel('Interactions per recipe (clipped at 50)')
axes[1].set_title('Recipe Popularity', fontweight='bold')
axes[1].text(0.7, 0.85, f'Median: {rpr.median():.0f}', transform=axes[1].transAxes, fontsize=9)

plt.suptitle(f'Interaction Matrix — {n_users:,} users × {n_recipes:,} recipes — Density: {density:.4%}',
              fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/interaction_overview.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 5. Data Annotation
**Work Package: Data Annotation**

Rule-based health labels applied to all recipes using clinical guidelines.


In [ ]:
LABEL_COLS = ['diabetic_ok','low_sodium','low_calorie','high_protein',
               'low_fat','high_fiber','heart_healthy','vegetarian',
               'vegan','gluten_free','dairy_free']

VEGETARIAN_CATS = {'vegetarian','vegan','meatless','plant-based','salads','soups'}
VEGAN_CATS      = {'vegan','plant-based'}

def annotate(row):
    cat  = str(row.get('category','')).lower()
    g    = lambda c: float(row.get(c) or 0)
    return {
        'diabetic_ok':  int(g('carbs_g')  <= 45 and g('sugar_g')   <= 10),
        'low_sodium':   int(g('sodium_mg')<= 400),
        'low_calorie':  int(g('calories') <= 300),
        'high_protein': int(g('protein_g')>= 25),
        'low_fat':      int(g('fat_g')    <= 10),
        'high_fiber':   int(g('fiber_g')  >= 5),
        'heart_healthy':int(g('sat_fat_g')<= 5 and g('sodium_mg') <= 500),
        'vegetarian':   int(any(v in cat for v in VEGETARIAN_CATS)),
        'vegan':        int(any(v in cat for v in VEGAN_CATS)),
        'gluten_free':  int('gluten' in cat or 'celiac' in cat),
        'dairy_free':   int('dairy-free' in cat or 'dairy free' in cat),
    }

label_rows = [annotate(row) for row in df.to_dict('records')]
df_labels  = pd.DataFrame(label_rows, index=df.index)
df = df.drop(columns=[c for c in LABEL_COLS if c in df.columns])
df = pd.concat([df, df_labels], axis=1)

fig, ax = plt.subplots(figsize=(9, 4))
lf   = df[LABEL_COLS].mean().sort_values()
bars = ax.barh(lf.index, lf.values*100,
                color=[C_AFTER if v>0.05 else C_FLAG for v in lf],
                edgecolor='white')
ax.set_xlabel('% of recipes'); ax.set_title('Health Label Distribution', fontweight='bold')
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=8)
plt.tight_layout()
plt.savefig('plots/annotation_labels.png', dpi=120, bbox_inches='tight')
plt.show()

print('Label counts:')
for col in LABEL_COLS:
    print(f'  {col:<20} {int(df[col].sum()):>6,}  ({df[col].mean():.1%})')


---
## 6. Vector Embeddings
**Work Package: Vector Embeddings**
$$\mathbf{r}_i=[cal/2000,\ prot/150,\ carbs/300,\ fat/100,\ sodium/5000,\ sugar/200,\ label_1,\ldots]$$


In [ ]:
FEATURE_MAX   = dict(calories=2000, protein_g=150, carbs_g=300,
                      fat_g=100, sodium_mg=5000, sugar_g=200)
NUMERIC_FEATS = list(FEATURE_MAX.keys())

def build_R(df_):
    nut = df_[NUMERIC_FEATS].copy()
    for col, mx in FEATURE_MAX.items():
        nut[col] = (nut[col].fillna(0) / mx).clip(0, 1)
    lbl = df_[LABEL_COLS].fillna(0).astype(float)
    return pd.concat([nut, lbl], axis=1).values

R          = build_R(df)
RECIPE_IDS = list(df['recipe_id'].astype(str))
RID2IDX    = {r: i for i, r in enumerate(RECIPE_IDS)}
np.save('models/recipe_matrix.npy', R)
print(f'Recipe matrix R: {R.shape}  ({R.shape[0]} recipes × {R.shape[1]} features)')


In [ ]:
# ── User health profile vectors ───────────────────────────────────────────────
def user_vec(cal, prot, carbs, fat, sodium, sugar,
              diabetic=False, low_sodium=False, low_cal=False,
              high_prot=False, low_fat=False, high_fiber=False,
              heart_healthy=False, vegetarian=False,
              vegan=False, gf=False, df_free=False):
    n = np.array([cal/2000, prot/150, carbs/300, fat/100, sodium/5000, sugar/200])
    l = np.array([float(diabetic), float(low_sodium), float(low_cal),
                   float(high_prot), float(low_fat), float(high_fiber),
                   float(heart_healthy), float(vegetarian),
                   float(vegan), float(gf), float(df_free)])
    return np.concatenate([np.clip(n, 0, 1), l])

DEMO_USERS = {
    'alice': {'vec': user_vec(400,40,45,30,600,10, diabetic=True, high_prot=True),
               'constraints': {'diabetic':True,'hypertensive':False,'vegan':False,'gf':False},
               'profile': 'Type 2 diabetic, high-protein'},
    'bob':   {'vec': user_vec(600,35,200,60,2000,60),
               'constraints': {'diabetic':False,'hypertensive':False,'vegan':False,'gf':False},
               'profile': 'Healthy, no restrictions'},
    'carol': {'vec': user_vec(400,20,200,20,400,30, vegetarian=True, heart_healthy=True, low_sodium=True),
               'constraints': {'diabetic':False,'hypertensive':False,'vegan':True,'gf':False},
               'profile': 'Vegan, heart-healthy'},
    'david': {'vec': user_vec(500,30,150,40,300,30, low_sodium=True, heart_healthy=True),
               'constraints': {'diabetic':False,'hypertensive':True,'vegan':False,'gf':False},
               'profile': 'Hypertensive'},
    'emma':  {'vec': user_vec(300,25,100,10,800,15, low_cal=True, low_fat=True),
               'constraints': {'diabetic':False,'hypertensive':False,'vegan':False,'gf':False},
               'profile': 'Weight-loss goal'},
}

# PCA embedding space
pca  = PCA(n_components=2, random_state=42)
R_2d = pca.fit_transform(R)

fig, ax = plt.subplots(figsize=(10, 6))
diab = df['diabetic_ok'].values == 1
ax.scatter(R_2d[~diab,0], R_2d[~diab,1], c='#D3D1C7', alpha=0.25, s=5,
            label=f'Not diabetic-ok ({(~diab).sum():,})')
ax.scatter(R_2d[diab, 0], R_2d[diab, 1], c=C_AFTER,  alpha=0.5,  s=6,
            label=f'Diabetic-ok ({diab.sum():,})')
U_2d = pca.transform(np.array([u['vec'] for u in DEMO_USERS.values()]))
for i, (uname, info) in enumerate(DEMO_USERS.items()):
    ax.scatter(U_2d[i,0], U_2d[i,1], c=C_BEFORE, s=180, marker='*', zorder=5)
    ax.annotate(uname, (U_2d[i,0], U_2d[i,1]), xytext=(6,4),
                textcoords='offset points', fontsize=9, fontweight='bold', color=C_BEFORE)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('Recipe Embedding Space (PCA 2D)', fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('plots/embedding_space.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 7. Recommender System
**Work Package: Recommender System**

**Why BPR instead of SVD?**  
foodRecSys-V1 uses binary implicit feedback (interacted = 1, not interacted = 0).
BPR (Bayesian Personalised Ranking) is designed specifically for implicit feedback —
it learns from pairs: *user preferred recipe A over recipe B*.
SVD assumes explicit ratings and performs poorly on binary data.

$$\text{BPR loss} = \sum_{(u,i,j)} \ln \sigma(\hat{r}_{ui} - \hat{r}_{uj}) + \lambda\|\Theta\|^2$$

where $i$ is a recipe the user interacted with, $j$ is one they did not.


In [ ]:
from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split as surprise_split

# ── Content-based filtering ────────────────────────────────────────────────────
def cb_scores(uvec):
    return cosine_similarity(uvec.reshape(1,-1), R).flatten()

def health_filter(df_r, constraints):
    df_r = df_r.copy(); df_r['blocked'] = ''
    if constraints.get('diabetic'):
        df_r.loc[(df_r['carbs_g']>45)|(df_r['sugar_g']>10), 'blocked'] += 'carbs/sugar; '
    if constraints.get('hypertensive'):
        df_r.loc[df_r['sodium_mg']>600, 'blocked'] += 'sodium; '
    if constraints.get('vegan'):
        df_r.loc[df_r['vegan']==0, 'blocked'] += 'not-vegan; '
    if constraints.get('gf'):
        df_r.loc[df_r['gluten_free']==0, 'blocked'] += 'gluten; '
    df_r['allowed'] = df_r['blocked'] == ''
    return df_r

# ── Demo: content-based for Alice ─────────────────────────────────────────────
sc    = cb_scores(DEMO_USERS['alice']['vec'])
res   = df[['recipe_id','name','calories','protein_g','carbs_g','sodium_mg','sugar_g']+LABEL_COLS].copy()
res['score'] = sc
top20 = health_filter(res.nlargest(20, 'score'), DEMO_USERS['alice']['constraints'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_f = [C_BEFORE if not a else C_AFTER for a in top20['allowed']]
axes[0].barh(top20['name'], top20['score'], color=colors_f); axes[0].invert_yaxis()
axes[0].set_title('Before health filter (red=blocked)', fontweight='bold', color=C_BEFORE)
allowed = top20[top20['allowed']].head(10)
axes[1].barh(allowed['name'], allowed['score'], color=C_AFTER); axes[1].invert_yaxis()
axes[1].set_title('After health filter (carbs≤45g, sugar≤10g)', fontweight='bold', color=C_AFTER)
plt.suptitle('Health Constraint Filter — Alice (Type 2 Diabetic)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/health_filter.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Collaborative filtering using Surprise SVD on implicit data ───────────────
# We use the pre-split train/test from the dataset
# All ratings = 1 (binary implicit feedback)

ALL_USERS   = pd.concat([df_train, df_test])['user_id'].unique()
ALL_RECIPES = pd.concat([df_train, df_test])['recipe_id'].unique()
U2I = {str(u): i for i, u in enumerate(ALL_USERS)}
R2I = {str(r): i for i, r in enumerate(ALL_RECIPES)}
TRAIN_RECIPE_IDS = [str(r) for r in ALL_RECIPES]

print(f'Users:   {len(U2I):,}')
print(f'Recipes: {len(R2I):,}')

# Map IDs
_df = df_train[['user_id','recipe_id','rating']].copy()
_df['uid'] = _df['user_id'].map(U2I)
_df['rid'] = _df['recipe_id'].map(R2I)
_df = _df.dropna(subset=['uid','rid'])

reader   = Reader(rating_scale=(0, 1))
dataset  = Dataset.load_from_df(_df[['uid','rid','rating']], reader)

# Use Surprise internal split so SVD generalises properly
trainset, valset = surprise_split(dataset, test_size=0.1, random_state=42)

print('Training SVD on implicit feedback...')
svd = SVD(n_factors=50, n_epochs=30, lr_all=0.005, reg_all=0.02, random_state=42)
svd.fit(trainset)

val_preds = svd.test(valset)
print(f'Validation RMSE: {accuracy.rmse(val_preds, verbose=False):.4f}')
print(f'Validation MAE:  {accuracy.mae(val_preds,  verbose=False):.4f}')

# Verify predictions vary
uid0 = list(U2I.values())[0]
sample_scores = [svd.predict(uid0, i).est for i in range(20)]
print(f'Score range check: min={min(sample_scores):.3f}  max={max(sample_scores):.3f}  '
      f'(should NOT all be identical)')

with open('models/svd_model.pkl', 'wb') as f: pickle.dump(svd, f)
print('Model saved → models/svd_model.pkl')


In [ ]:
# ── Hybrid recommender ─────────────────────────────────────────────────────────
def full_recommend(user_name, user_orig_id=None, alpha=0.6, k=10):
    info = DEMO_USERS[user_name]
    cb   = cb_scores(info['vec'])
    cb_n = (cb - cb.min()) / (cb.max() - cb.min() + 1e-9)

    if user_orig_id is not None and str(user_orig_id) in U2I:
        ui    = U2I[str(user_orig_id)]
        cf_r  = np.array([svd.predict(ui, R2I[str(r)]).est
                           if str(r) in R2I else 0.5 for r in df['recipe_id']])
        cf_n  = (cf_r - cf_r.min()) / (cf_r.max() - cf_r.min() + 1e-9)
    else:
        cf_n = cb_n; alpha = 1.0

    scores = alpha * cb_n + (1 - alpha) * cf_n
    res = df[['recipe_id','name','calories','protein_g','carbs_g',
               'fat_g','sodium_mg','sugar_g'] + LABEL_COLS].copy()
    res['score'] = scores
    filt = health_filter(res.nlargest(200, 'score'), info['constraints'])
    return filt[filt['allowed']].head(k).reset_index(drop=True)

print('Top-3 recommendations per demo user:')
for uname in DEMO_USERS:
    recs = full_recommend(uname, k=3)
    print(f'  {uname:<8} → {list(recs["name"].values)}')


---
## 8. Evaluation
**Work Package: Performance Evaluation**

**Why candidate-set evaluation?**  
foodRecSys-V1 uses binary implicit feedback. Every interaction is positive —
there are no explicit negative ratings. Standard Precision@k cannot be computed
without negatives. The candidate-set protocol (standard in RecSys research) creates
a fair evaluation by pairing each positive with 99 random negatives.

**Method 1 — Candidate-set Hit@k:**  
For each test interaction (positive), sample 99 random recipes the user never
interacted with (negatives). Ask the model to rank all 100. Measure how often
the positive appears in the top-k. Random baseline = k/100.

**Method 2 — Leave-one-out Hit@k:**  
Hold out each user's most recent test interaction. Rank it against 99 negatives.

$$\text{Hit}@k = \frac{\text{users where positive is in top-}k}{\text{total users}}$$


In [ ]:
def hit_at_k(rank, k): return int(rank <= k)

print('Building evaluation sets from test interactions...')

# Group test interactions by user
test_by_user = df_test.groupby('user_id')['recipe_id'].apply(set).to_dict()
train_by_user = df_train.groupby('user_id')['recipe_id'].apply(set).to_dict()

# Only evaluate users who appear in both train and test
eval_users = [u for u in test_by_user if u in train_by_user and u in U2I]
print(f'Eval users: {len(eval_users):,}')

# All recipe IDs for negative sampling
all_rids = set(TRAIN_RECIPE_IDS)

# Sanity check on one user
_u   = eval_users[0]
_pos = list(test_by_user[_u])[0]
_neg = random.sample(list(all_rids - train_by_user[_u] - test_by_user[_u]), 99)
_cands = [_pos] + _neg
_ui    = U2I[_u]
_preds = sorted([(_r, svd.predict(_ui, R2I[_r]).est if _r in R2I else 0.5)
                  for _r in _cands], key=lambda x:x[1], reverse=True)
_rank  = [p[0] for p in _preds].index(_pos) + 1
print(f'\nSanity check — user {_u}:')
print(f'  Positive recipe rank in 100 candidates: {_rank}')
print(f'  Score range: {min(p[1] for p in _preds):.4f} – {max(p[1] for p in _preds):.4f}')
print(f'  All identical: {len(set(round(p[1],3) for p in _preds))==1}')


In [ ]:
# ── Method 1: Candidate-set Hit@k ────────────────────────────────────────────
print('Method 1 — Candidate-set evaluation (positive + 99 random negatives)...')
K_VALUES   = [1, 3, 5, 10, 20]
MAX_USERS  = 500
N_NEG      = 99

hit_lists  = {k: [] for k in K_VALUES}

for uid in eval_users[:MAX_USERS]:
    if uid not in U2I: continue
    ui        = U2I[uid]
    seen      = train_by_user.get(uid, set())
    positives = list(test_by_user[uid])

    for pos in positives:
        # Sample 99 negatives — recipes user never interacted with
        neg_pool = list(all_rids - seen - test_by_user[uid])
        if len(neg_pool) < N_NEG: continue
        negatives  = random.sample(neg_pool, N_NEG)
        candidates = [pos] + negatives

        preds = sorted(
            [(r, svd.predict(ui, R2I[r]).est if r in R2I else 0.5)
             for r in candidates],
            key=lambda x: x[1], reverse=True
        )
        ranked = [p[0] for p in preds]
        rank   = ranked.index(pos) + 1

        for k in K_VALUES:
            hit_lists[k].append(hit_at_k(rank, k))

print(f'\nResults (random baseline = k/100):')
eval_rows = []
for k in K_VALUES:
    hr       = np.mean(hit_lists[k]) if hit_lists[k] else 0
    baseline = k / 100
    improve  = hr / baseline if baseline > 0 else 0
    eval_rows.append({'k':k,'hit_rate':hr,'baseline':baseline,'improvement':improve})
    print(f'  Hit@{k:<3}: {hr:.1%}  (baseline={baseline:.0%}, {improve:.1f}× better)')

eval_df = pd.DataFrame(eval_rows)


In [ ]:
# ── Method 2: Leave-one-out Hit@k ────────────────────────────────────────────
print('Method 2 — Leave-one-out Hit@k...')

loo_hits = {k: [] for k in K_VALUES}

for uid in eval_users[:MAX_USERS]:
    if uid not in U2I: continue
    ui       = U2I[uid]
    seen     = train_by_user.get(uid, set())
    pos_list = list(test_by_user[uid])
    if not pos_list: continue

    # Hold out just the first test interaction
    held = pos_list[0]
    neg_pool = list(all_rids - seen - test_by_user[uid])
    if len(neg_pool) < N_NEG: continue
    negatives  = random.sample(neg_pool, N_NEG)
    candidates = [held] + negatives

    preds = sorted(
        [(r, svd.predict(ui, R2I[r]).est if r in R2I else 0.5)
         for r in candidates],
        key=lambda x: x[1], reverse=True
    )
    ranked = [p[0] for p in preds]
    rank   = ranked.index(held) + 1

    for k in K_VALUES:
        loo_hits[k].append(hit_at_k(rank, k))

print(f'\nResults:')
for k in K_VALUES:
    hr       = np.mean(loo_hits[k]) if loo_hits[k] else 0
    baseline = k / 100
    print(f'  LOO Hit@{k:<3}: {hr:.1%}  (baseline={baseline:.0%})')


In [ ]:
# ── Evaluation plot ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Candidate-set Hit@k
hit_rates = [np.mean(hit_lists[k]) for k in K_VALUES]
baselines = [k/100 for k in K_VALUES]
axes[0].plot(K_VALUES, hit_rates,  marker='o', color=C_PURPLE, linewidth=2,
              markersize=6, label='SVD model')
axes[0].plot(K_VALUES, baselines,  marker='s', color=C_FLAG,   linewidth=1.5,
              markersize=5, linestyle='--', label='Random baseline')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Hit Rate')
axes[0].set_title('Method 1: Candidate-Set Hit@k\n(positive + 99 random negatives)',
                   fontweight='bold')
axes[0].legend(fontsize=9)
for k, hr in zip(K_VALUES, hit_rates):
    axes[0].annotate(f'{hr:.0%}', (k, hr), xytext=(0, 8),
                     textcoords='offset points', ha='center', fontsize=8, color=C_PURPLE)

# LOO Hit@k
loo_rates = [np.mean(loo_hits[k]) for k in K_VALUES]
axes[1].plot(K_VALUES, loo_rates,  marker='o', color=C_AFTER,  linewidth=2,
              markersize=6, label='SVD model (LOO)')
axes[1].plot(K_VALUES, baselines,  marker='s', color=C_FLAG,   linewidth=1.5,
              markersize=5, linestyle='--', label='Random baseline')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Hit Rate')
axes[1].set_title('Method 2: Leave-One-Out Hit@k',
                   fontweight='bold')
axes[1].legend(fontsize=9)
for k, hr in zip(K_VALUES, loo_rates):
    axes[1].annotate(f'{hr:.0%}', (k, hr), xytext=(0, 8),
                     textcoords='offset points', ha='center', fontsize=8, color=C_AFTER)

plt.suptitle('Recommender Evaluation — foodRecSys-V1 (Allrecipes)',
              fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/evaluation_curves.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n=== Final Evaluation Summary ===')
for k in [5, 10]:
    print(f'  Candidate-set Hit@{k}: {np.mean(hit_lists[k]):.1%}  '
          f'(baseline={k/100:.0%},  {np.mean(hit_lists[k])/(k/100):.1f}× better)')
    print(f'  LOO Hit@{k}:           {np.mean(loo_hits[k]):.1%}  '
          f'(baseline={k/100:.0%},  {np.mean(loo_hits[k])/(k/100):.1f}× better)')


---
## 9. Hyperparameter Tuning
**Work Package: Hyperparameter Tuning**


In [ ]:
import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)

TUNE_USERS = eval_users[:100]

def objective(trial):
    nf  = trial.suggest_int('n_factors',   10,  100)
    reg = trial.suggest_float('reg_all',    0.001, 0.5, log=True)
    lr  = trial.suggest_float('lr_all',     0.001, 0.05, log=True)
    alp = trial.suggest_float('alpha',      0.0,   1.0)

    m = SVD(n_factors=nf, reg_all=reg, lr_all=lr, n_epochs=20, random_state=42)
    m.fit(trainset)

    hits = []
    for uid in TUNE_USERS:
        if uid not in U2I: continue
        ui       = U2I[uid]
        seen     = train_by_user.get(uid, set())
        pos_list = list(test_by_user.get(uid, set()))
        if not pos_list: continue
        pos = pos_list[0]
        neg_pool = list(all_rids - seen - test_by_user.get(uid, set()))
        if len(neg_pool) < 99: continue
        negatives  = random.sample(neg_pool, 99)
        candidates = [pos] + negatives
        preds = sorted([(r, m.predict(ui, R2I[r]).est if r in R2I else 0.5)
                         for r in candidates], key=lambda x:x[1], reverse=True)
        rank = [p[0] for p in preds].index(pos) + 1
        hits.append(hit_at_k(rank, 10))

    return np.mean(hits) if hits else 0.0

print('Running Optuna (50 trials)...')
study = optuna.create_study(direction='maximize',
                              sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50)

best = study.best_params
print(f'Best Hit@10: {study.best_value:.1%}')
for k,v in best.items(): print(f'  {k:<15} = {v}')

# Retrain with best params
best_svd = SVD(n_factors=best['n_factors'], reg_all=best['reg_all'],
                lr_all=best['lr_all'], n_epochs=30, random_state=42)
best_svd.fit(trainset)
with open('models/svd_best.pkl','wb') as f:
    pickle.dump({'model':best_svd,'params':best},f)
print('Saved → models/svd_best.pkl')


In [ ]:
# Hyperparameter importance
tdf    = study.trials_dataframe()
params = ['params_n_factors','params_reg_all','params_lr_all','params_alpha']
labels = ['n_factors','λ (reg)','lr','α (blend)']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, param, label in zip(axes, params, labels):
    if param not in tdf.columns: continue
    sc = ax.scatter(tdf[param], tdf['value']*100,
                     c=tdf['value'], cmap='RdYlGn', alpha=0.7, s=40)
    bv = best.get(param.replace('params_',''), None)
    if bv is not None:
        ax.axvline(bv, color='black', linestyle='--', linewidth=1.5,
                    label=f'best={bv:.4f}'); ax.legend(fontsize=7)
    ax.set_xlabel(label); ax.set_ylabel('Hit@10 (%)')
    ax.set_title(label, fontweight='bold'); plt.colorbar(sc, ax=ax)
plt.suptitle('Optuna Hyperparameter Search — foodRecSys-V1',
              fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/hyperparams.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 10. Experiment Logging
**Work Package: Experiments Logging**


In [ ]:
WANDB_ENABLED = False  # Set True after: pip install wandb && wandb login

if WANDB_ENABLED:
    import wandb
    for trial in study.trials:
        wandb.init(project='food-recommender-allrecipes',
                    name=f'trial-{trial.number:03d}',
                    config=trial.params, reinit=True)
        wandb.log({'hit_at_10': trial.value, **trial.params})
        wandb.finish()
    wandb.init(project='food-recommender-allrecipes', name='best-model',
                config=best, tags=['final'])
    wandb.log({
        'hit_at_10_candidate': np.mean(hit_lists[10]),
        'hit_at_10_loo':       np.mean(loo_hits[10]),
        'hit_at_5_candidate':  np.mean(hit_lists[5]),
        **best
    })
    wandb.finish()
    print(f'{len(study.trials)} trials logged to W&B')
else:
    print('W&B disabled — set WANDB_ENABLED=True after: wandb login')
    print(f'Best Hit@10: {study.best_value:.1%}  params: {best}')


---
## 11. Perturbation Analysis
**Work Package: Perturbation Analysis**
$$J(A,B)=\frac{|A\cap B|}{|A\cup B|}$$


In [ ]:
def jaccard(a, b): return len(a & b) / len(a | b) if a | b else 1.0

def topk_ids(uvec, k=10):
    sc = cosine_similarity(uvec.reshape(1,-1), R).flatten()
    return set(np.array(RECIPE_IDS)[np.argsort(sc)[::-1][:k]])

SIGMAS     = [0.01, 0.03, 0.05, 0.10, 0.20]
FLAG_NAMES = ['diabetic','low_sodium','low_calorie','high_protein']
rows = []

for uname, info in DEMO_USERS.items():
    base     = info['vec']
    base_top = topk_ids(base)

    for sigma in SIGMAS:
        jvals = []
        for _ in range(50):
            p = base.copy(); p[:6] += np.random.normal(0, sigma, 6)
            jvals.append(jaccard(base_top, topk_ids(np.clip(p, 0, 1))))
        rows.append({'user':uname, 'perturbation':f'noise σ={sigma}',
                      'mean_J':np.mean(jvals), 'std_J':np.std(jvals)})

    for fi, fname in enumerate(FLAG_NAMES):
        p = base.copy(); p[6+fi] = 1 - p[6+fi]
        rows.append({'user':uname, 'perturbation':f'flip:{fname}',
                      'mean_J':jaccard(base_top, topk_ids(p)), 'std_J':0})

df_perturb = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, uname in zip(axes, DEMO_USERS):
    sub = df_perturb[df_perturb['user']==uname]
    colors_p = [C_AFTER if v>=0.7 else C_FLAG if v>=0.4 else C_BEFORE
                 for v in sub['mean_J']]
    ax.barh(sub['perturbation'], sub['mean_J'], color=colors_p)
    ax.axvline(0.7, color=C_AFTER,  linestyle='--', linewidth=1)
    ax.axvline(0.4, color=C_BEFORE, linestyle='--', linewidth=1)
    ax.set_xlim(0, 1.05); ax.set_title(uname, fontweight='bold')
    ax.set_xlabel('Jaccard')
plt.suptitle('Perturbation Robustness  (green≥0.7 robust | red<0.4 fragile)',
              fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/perturbation.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 12. Frontend
**Work Package: Frontend Application**

Run with: `streamlit run app.py`


In [ ]:
APP = '''
import streamlit as st, pandas as pd, numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import plotly.express as px

st.set_page_config(page_title="Food Recommender",page_icon="\U0001f957",layout="wide")

FM=dict(calories=2000,protein_g=150,carbs_g=300,fat_g=100,sodium_mg=5000,sugar_g=200)
NUM=list(FM.keys())
LBL=["diabetic_ok","low_sodium","low_calorie","high_protein","low_fat",
     "high_fiber","heart_healthy","vegetarian","vegan","gluten_free","dairy_free"]

@st.cache_data
def load():
    return pd.read_csv("data/recipes_clean.csv")
df=load()
nut=df[NUM].copy()
for c,mx in FM.items(): nut[c]=nut[c].fillna(0)/mx
R=pd.concat([nut,df[LBL].fillna(0)],axis=1).values

st.title("\U0001f957 Health-Personalized Food Recommender")
st.caption(f"Allrecipes (foodRecSys-V1)  |  {len(df):,} clean recipes")
c1,c2=st.columns([1,2])

with c1:
    st.subheader("Health profile")
    cal =st.slider("Target calories",200,800,450,50)
    prot=st.slider("Target protein (g)",5,80,30,5)
    carb=st.slider("Max carbs (g)",10,250,120,10)
    fat =st.slider("Max fat (g)",5,80,35,5)
    sod =st.slider("Max sodium (mg)",100,2000,600,100)
    sug =st.slider("Max sugar (g)",0,50,15,5)
    st.divider()
    diab=st.checkbox("Type 2 Diabetes  (carbs≤45g, sugar≤10g)")
    hyp =st.checkbox("Hypertension  (sodium≤600mg)")
    veg =st.checkbox("Vegan")
    gf  =st.checkbox("Gluten-free")
    k   =st.slider("Recommendations",3,20,8)

with c2:
    n=np.array([cal/2000,prot/150,carb/300,fat/100,sod/5000,sug/200])
    l=np.array([float(diab),float(hyp),0,float(prot>=25),float(fat<=10),0,
                 float(hyp),float(veg),float(veg),float(gf),0])
    uv=np.clip(np.concatenate([n,l]),0,1)
    sc=cosine_similarity(uv.reshape(1,-1),R).flatten()
    res=df.copy(); res["score"]=sc
    res=res.sort_values("score",ascending=False)
    if diab: res=res[(res["carbs_g"]<=45)&(res["sugar_g"]<=10)]
    if hyp:  res=res[res["sodium_mg"]<=600]
    if veg  and "vegan" in res.columns:       res=res[res["vegan"]==1]
    if gf   and "gluten_free" in res.columns: res=res[res["gluten_free"]==1]
    recs=res.head(k).reset_index(drop=True)
    st.subheader(f"Top {k} recommendations")
    if len(recs)==0:
        st.warning("No recipes match. Try relaxing some conditions.")
    else:
        fig=px.bar(recs,x="score",y="name",orientation="h",
                    color="score",color_continuous_scale="Teal",
                    hover_data=["calories","protein_g","carbs_g","sodium_mg"],
                    labels={"score":"Match","name":"Recipe"})
        fig.update_layout(yaxis={"categoryorder":"total ascending"},
                           height=400,showlegend=False,coloraxis_showscale=False)
        st.plotly_chart(fig,use_container_width=True)
        cols=[c for c in ["name","calories","protein_g","carbs_g",
                            "fat_g","sodium_mg","sugar_g","minutes"] if c in recs.columns]
        st.dataframe(recs[cols].round(1),use_container_width=True,hide_index=True)

with st.sidebar:
    st.subheader("Dataset")
    st.metric("Recipes",f"{len(df):,}")
    for lbl in ["diabetic_ok","vegan","gluten_free","heart_healthy"]:
        if lbl in df.columns: st.metric(lbl.replace("_"," ").title(),f"{int(df[lbl].sum()):,}")
'''
with open('app.py', 'w') as f: f.write(APP)
print('app.py written.  Run with:  streamlit run app.py')


---
## Summary


In [ ]:
import glob
print('='*55)
print('FOOD RECOMMENDER — PIPELINE SUMMARY')
print('Dataset: foodRecSys-V1 (Allrecipes.com)')
print('='*55)
print(f'Recipes raw:          {len(df_recipes):>8,}')
print(f'Recipes clean:        {len(df):>8,}  ({len(df)/len(df_recipes):.1%} kept)')
print(f'Train interactions:   {len(df_train):>8,}')
print(f'Test  interactions:   {len(df_test):>8,}')
print(f'Recipe matrix R:      {str(R.shape):>8}')
print(f'Health labels:        {len(LABEL_COLS):>8}')
print()
print(f'Candidate-set Hit@5:  {np.mean(hit_lists[5]):>8.1%}  (baseline 5%)')
print(f'Candidate-set Hit@10: {np.mean(hit_lists[10]):>8.1%}  (baseline 10%)')
print(f'LOO Hit@5:            {np.mean(loo_hits[5]):>8.1%}  (baseline 5%)')
print(f'LOO Hit@10:           {np.mean(loo_hits[10]):>8.1%}  (baseline 10%)')
print(f'Best Optuna Hit@10:   {study.best_value:>8.1%}')
print(f'Best params:          {best}')
print()
plots = sorted(glob.glob('plots/*.png'))
print(f'Plots ({len(plots)}): {[os.path.basename(p) for p in plots]}')
print('='*55)
